In [1]:
!pip -q install datasets qdrant-client transformers torch pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os

if os.path.exists('MedRAG.zip'):
    !unzip -o MedRAG.zip
else:
    print('MedRAG.zip not found.')

In [ ]:
!unzip /content/medical_rag_cache.zip

In [2]:
import os
import time
import uuid
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# -------------------------
# Config
# -------------------------
CORPUS_SIZE = 50000      # try 20000 first if Colab RAM is tight, then 50000, 100000
EVAL_SIZE = 5000          # later 1000
BATCH_SIZE = 32
QDRANT_PATH = "/content/qdrant_pubmedqa_medcpt"
COLLECTION_NAME = "pubmedqa_medcpt_dense"

ARTICLE_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
QUERY_MODEL_NAME   = "ncbi/MedCPT-Query-Encoder"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [12]:
#/content/content/medical_rag_cache
corpus_df = pd.read_parquet("/content/content/medical_rag_cache/corpus.parquet")
eval_df   = pd.read_parquet("/content/content/medical_rag_cache/eval.parquet")
doc_vecs  = np.load("/content/content/medical_rag_cache/doc_vecs.npy")

In [13]:
# ============================================================
# QDRANT UPLOAD FIX
# Keep original doc_id in payload, use integer point IDs
# ============================================================

import numpy as np
from qdrant_client.models import PointStruct



# Create an integer point_id for Qdrant, preserve original doc_id in payload
corpus_df = corpus_df.copy().reset_index(drop=True)
corpus_df["point_id"] = np.arange(1, len(corpus_df) + 1, dtype=np.int64)

# Optional lookup maps for later
point_to_doc = dict(zip(corpus_df["point_id"], corpus_df["doc_id"]))
doc_to_point = dict(zip(corpus_df["doc_id"], corpus_df["point_id"]))

points = []
for row, vec in zip(corpus_df.to_dict("records"), doc_vecs):
    points.append(
        PointStruct(
            id=int(row["point_id"]),   # <-- integer ID, valid for Qdrant
            vector=vec.tolist(),
            payload={
                "doc_id": str(row["doc_id"]),   # keep original ID here
                "title": row["title"],
                "text": row["text"][:4000],
                "source": row["source"],
                "final_decision": row["final_decision"],
            }
        )
    )

# Fresh collection
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=int(doc_vecs.shape[1]), distance=Distance.COSINE)
)

# Upload in smaller batches
for i in range(0, len(points), 128):
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i+128]
    )
    if i % 1024 == 0:
        print("Uploaded:", min(i + 128, len(points)), "/", len(points))

print("Upload complete:", len(points))

Uploaded: 128 / 51000
Uploaded: 1152 / 51000
Uploaded: 2176 / 51000
Uploaded: 3200 / 51000
Uploaded: 4224 / 51000
Uploaded: 5248 / 51000
Uploaded: 6272 / 51000
Uploaded: 7296 / 51000
Uploaded: 8320 / 51000
Uploaded: 9344 / 51000
Uploaded: 10368 / 51000
Uploaded: 11392 / 51000
Uploaded: 12416 / 51000
Uploaded: 13440 / 51000
Uploaded: 14464 / 51000
Uploaded: 15488 / 51000
Uploaded: 16512 / 51000
Uploaded: 17536 / 51000
Uploaded: 18560 / 51000
Uploaded: 19584 / 51000


/tmp/ipykernel_3297/2203251411.py:48: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20096 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Uploaded: 20608 / 51000
Uploaded: 21632 / 51000
Uploaded: 22656 / 51000
Uploaded: 23680 / 51000
Uploaded: 24704 / 51000
Uploaded: 25728 / 51000
Uploaded: 26752 / 51000
Uploaded: 27776 / 51000
Uploaded: 28800 / 51000
Uploaded: 29824 / 51000
Uploaded: 30848 / 51000
Uploaded: 31872 / 51000
Uploaded: 32896 / 51000
Uploaded: 33920 / 51000
Uploaded: 34944 / 51000
Uploaded: 35968 / 51000
Uploaded: 36992 / 51000
Uploaded: 38016 / 51000
Uploaded: 39040 / 51000
Uploaded: 40064 / 51000
Uploaded: 41088 / 51000
Uploaded: 42112 / 51000
Uploaded: 43136 / 51000
Uploaded: 44160 / 51000
Uploaded: 45184 / 51000
Uploaded: 46208 / 51000
Uploaded: 47232 / 51000
Uploaded: 48256 / 51000
Uploaded: 49280 / 51000
Uploaded: 50304 / 51000
Upload complete: 51000


In [14]:
len(eval_df)

1000

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [17]:
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# load MedCPT query encoder
query_model_name = "ncbi/MedCPT-Query-Encoder"
query_tok = AutoTokenizer.from_pretrained(query_model_name)
query_model = AutoModel.from_pretrained(query_model_name).to(device)
query_model.eval()

# prepare queries
queries = eval_df["query"].astype(str).tolist()
gold_doc_ids = eval_df["gold_doc_id"].astype(str).tolist()

# encode queries
query_vectors_list = []

with torch.no_grad():
    for start in tqdm(range(0, len(queries), 32), desc="Encoding queries"):
        end = start + 32
        batch_queries = queries[start:end]

        encoded = query_tok(
            batch_queries,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        output = query_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        batch_vectors = output.last_hidden_state[:, 0, :]
        batch_vectors = torch.nn.functional.normalize(batch_vectors, p=2, dim=1)

        batch_vectors = batch_vectors.cpu().numpy().astype(np.float32)
        query_vectors_list.append(batch_vectors)

query_vectors = np.vstack(query_vectors_list)

# search and collect ranks
ranks = []

for i in tqdm(range(len(queries)), desc="Searching Qdrant"):
    gold_doc_id = gold_doc_ids[i]
    query_vector = query_vectors[i].tolist()

    result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=10,
        with_payload=True
    )

    hits = result.points

    retrieved_doc_ids = []
    for hit in hits:
        doc_id = str(hit.payload["doc_id"])
        retrieved_doc_ids.append(doc_id)

    rank = None
    for position, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == gold_doc_id:
            rank = position
            break

    ranks.append(rank)

# metrics
total_queries = len(ranks)

hit_at_1 = 0
hit_at_5 = 0
hit_at_10 = 0
mrr_total = 0.0

for rank in ranks:
    if rank is not None:
        if rank <= 1:
            hit_at_1 += 1
        if rank <= 5:
            hit_at_5 += 1
        if rank <= 10:
            hit_at_10 += 1

        mrr_total += 1.0 / rank

hit_at_1 = hit_at_1 / total_queries
hit_at_5 = hit_at_5 / total_queries
hit_at_10 = hit_at_10 / total_queries
mrr = mrr_total / total_queries

print("\nDense Retriever Benchmark")
print("Queries :", total_queries)
print("Hit@1   :", round(hit_at_1, 4))
print("Hit@5   :", round(hit_at_5, 4))
print("Hit@10  :", round(hit_at_10, 4))
print("MRR     :", round(mrr, 4))

Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/32 [00:00<?, ?it/s]

Searching Qdrant:   0%|          | 0/1000 [00:00<?, ?it/s]


Dense Retriever Benchmark
Queries : 1000
Hit@1   : 0.924
Hit@5   : 0.993
Hit@10  : 0.997
MRR     : 0.9539


In [18]:
!pip install rank-bm25

In [19]:
import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

# prepare documents
doc_texts = (corpus_df["title"].fillna("") + " " + corpus_df["text"].fillna("")).astype(str).tolist()
doc_ids = corpus_df["doc_id"].astype(str).tolist()

# simple tokenization
tokenized_docs = []
for text in doc_texts:
    tokens = text.lower().split()
    tokenized_docs.append(tokens)

# build BM25 index
bm25 = BM25Okapi(tokenized_docs)

# prepare queries
queries = eval_df["query"].astype(str).tolist()
gold_doc_ids = eval_df["gold_doc_id"].astype(str).tolist()

# search and collect ranks
ranks = []

for i in tqdm(range(len(queries)), desc="BM25 search"):
    query = queries[i]
    gold_doc_id = gold_doc_ids[i]

    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:10]

    retrieved_doc_ids = []
    for idx in top_indices:
        retrieved_doc_ids.append(doc_ids[idx])

    rank = None
    for position, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == gold_doc_id:
            rank = position
            break

    ranks.append(rank)

# metrics
total_queries = len(ranks)

hit_at_1 = 0
hit_at_5 = 0
hit_at_10 = 0
mrr_total = 0.0

for rank in ranks:
    if rank is not None:
        if rank <= 1:
            hit_at_1 += 1
        if rank <= 5:
            hit_at_5 += 1
        if rank <= 10:
            hit_at_10 += 1

        mrr_total += 1.0 / rank

hit_at_1 = hit_at_1 / total_queries
hit_at_5 = hit_at_5 / total_queries
hit_at_10 = hit_at_10 / total_queries
mrr = mrr_total / total_queries

print("\nBM25 Benchmark")
print("Queries :", total_queries)
print("Hit@1   :", round(hit_at_1, 4))
print("Hit@5   :", round(hit_at_5, 4))
print("Hit@10  :", round(hit_at_10, 4))
print("MRR     :", round(mrr, 4))

BM25 search:   0%|          | 0/1000 [00:00<?, ?it/s]


BM25 Benchmark
Queries : 1000
Hit@1   : 0.999
Hit@5   : 1.0
Hit@10  : 1.0
MRR     : 0.9995
